# The Isolated Agent Terminal Sandbox

Build a tool-using agent with real guardrails: a tool registry, a validation layer, a step cap, and human approval for risky actions. **Runtime:** CPU. Needs an LLM API key.

## 1. Setup
Set your API key (this example uses the OpenAI SDK; adapt to any provider).

In [ ]:
!pip -q install openai
import os, json
os.environ['OPENAI_API_KEY'] = 'sk-...'  # <-- paste your key
from openai import OpenAI
client = OpenAI()

## 2. Define safe tools
Each tool is a plain function. Note what we deliberately leave OUT (no shell, no delete).

In [ ]:
import math

def calculator(expr):
    # SECURITY: eval is intentionally fenced by a strict math-only allow-list so
    # nothing but arithmetic can run. A real app should use a proper expression
    # parser (e.g. ast.literal_eval or a math grammar) rather than eval at all.
    if not all(c in '0123456789+-*/(). ' for c in expr):
        return 'rejected: invalid characters'
    return str(eval(expr))

def word_count(text):
    return str(len(text.split()))

TOOLS = {'calculator': calculator, 'word_count': word_count}

## 3. A validation layer
Every tool call is checked before it runs — the agent can never call something not on the allow-list.

In [ ]:
def run_tool(name, arg):
    if name not in TOOLS:
        return f'rejected: unknown tool {name}'
    try:
        return TOOLS[name](arg)
    except Exception as e:
        return f'error: {e}'  # tools never crash the loop

print(run_tool('calculator', '6 * 7'))
print(run_tool('calculator', 'import os'))  # blocked

## 4. The agent loop with a step cap
The model proposes a tool + argument as JSON; we validate, execute, and feed the result back — capped so it can never run forever.

In [ ]:
def agent(goal, max_steps=5):
    history = [{'role': 'system', 'content': 'You can call tools. Reply ONLY as JSON: {"tool":..., "arg":...} or {"final":...}. Tools: calculator, word_count.'},
               {'role': 'user', 'content': goal}]
    for step in range(max_steps):
        reply = client.chat.completions.create(model='gpt-4o-mini', messages=history).choices[0].message.content
        try:
            action = json.loads(reply)
        except Exception:
            return reply
        if 'final' in action:
            return action['final']
        result = run_tool(action.get('tool'), action.get('arg'))
        print(f'step {step}: {action} -> {result}')
        history.append({'role': 'assistant', 'content': reply})
        history.append({'role': 'user', 'content': f'Tool result: {result}'})
    return 'stopped: hit step cap'

## 5. Run it

In [ ]:
print(agent('What is 123 * 45, and how many words are in this sentence?'))

## Homework
1. Add a `requires_approval` flag to risky tools and prompt for a y/n before running them.
2. Add a tool that reads files, sandboxed to a single folder (path-traversal safe).
3. Log every action to a list and print the full trace at the end.
4. Try to jailbreak your own agent — then patch the hole you find.